# Investigating the 2023 Regulatory Peak During the Biden Administration

This tutorial explores why 2023 stands out as the single largest year for federal regulatory docket creation in the entire dataset (2000-present). We'll use the pre-aggregated data from the Spicy Regs Dashboard to quantify and visualize the anomaly.

**Key question:** Why did 2023 see ~50,000 dockets (2.7x the prior year) and ~386 million public comments (18x the prior year)?

## Data Sources
- `public/data/dockets-by-year.json` - Docket counts by year and regulatory cluster
- `public/data/comments-by-year.json` - Public comment totals by year
- `public/data/agency-activity.json` - Per-agency docket counts by administration
- `public/data/clusters.json` - Cluster breakdowns by administration

All data originates from [regulations.gov](https://www.regulations.gov) via the Mirrulations ETL pipeline.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# Load all dashboard data files
DATA_DIR = Path("../public/data")

with open(DATA_DIR / "dockets-by-year.json") as f:
    dockets_by_year = json.load(f)

with open(DATA_DIR / "comments-by-year.json") as f:
    comments_by_year = json.load(f)

with open(DATA_DIR / "agency-activity.json") as f:
    agency_activity = json.load(f)

with open(DATA_DIR / "clusters.json") as f:
    clusters = json.load(f)

df_dockets = pd.DataFrame(dockets_by_year)
df_comments = pd.DataFrame(comments_by_year)
df_agency = pd.DataFrame(agency_activity)

print(f"Loaded {len(df_dockets)} years of docket data ({df_dockets.year.min()}-{df_dockets.year.max()})")
print(f"Loaded {len(df_comments)} years of comment data")
print(f"Loaded {len(df_agency)} agency-admin records")

## 1. The Big Picture: Total Dockets Over Time

Let's start by visualizing total docket creation by year to see how 2023 compares to the historical baseline.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = ['#ef4444' if y == 2023 else '#6366f1' for y in df_dockets['year']]
ax.bar(df_dockets['year'], df_dockets['total'], color=colors, edgecolor='white', linewidth=0.5)

ax.set_title('Total Federal Regulatory Dockets by Year', fontsize=16, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Dockets')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# Annotate the 2023 peak
peak_val = df_dockets.loc[df_dockets.year == 2023, 'total'].values[0]
ax.annotate(f'2023 Peak\n{peak_val:,} dockets', xy=(2023, peak_val),
            xytext=(2018, peak_val * 0.9), fontsize=11, fontweight='bold', color='#ef4444',
            arrowprops=dict(arrowstyle='->', color='#ef4444', lw=1.5))

# Biden shading
ax.axvspan(2021, 2025, alpha=0.08, color='blue', label='Biden Admin')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

# Stats
median_total = df_dockets.loc[df_dockets.year.between(2005, 2022), 'total'].median()
print(f"2023 total: {peak_val:,} dockets")
print(f"Median (2005-2022): {median_total:,.0f} dockets")
print(f"2023 is {peak_val / median_total:.1f}x the historical median")

## 2. Which Regulatory Clusters Drove the 2023 Spike?

The dashboard categorizes agencies into 6 clusters: environment, health, finance, defense, transportation, and communications (plus "other"). Let's break down 2023 by cluster and compare to prior years.

In [ ]:
cluster_cols = ['environment', 'health', 'finance', 'defense', 'transportation', 'communications', 'other']
cluster_colors = {
    'health': '#ef4444',
    'other': '#8b5cf6',
    'transportation': '#3b82f6',
    'environment': '#22c55e',
    'finance': '#f59e0b',
    'defense': '#64748b',
    'communications': '#ec4899'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Stacked area chart
ax1 = axes[0]
bottom = None
for col in cluster_cols:
    vals = df_dockets[col].values
    if bottom is None:
        ax1.fill_between(df_dockets['year'], 0, vals, label=col.title(), color=cluster_colors[col], alpha=0.8)
        bottom = vals.copy()
    else:
        ax1.fill_between(df_dockets['year'], bottom, bottom + vals, label=col.title(), color=cluster_colors[col], alpha=0.8)
        bottom = bottom + vals

ax1.set_title('Docket Volume by Cluster (Stacked)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Year')
ax1.set_ylabel('Dockets')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax1.legend(loc='upper left', fontsize=8)
ax1.axvline(2023, color='red', linestyle='--', alpha=0.5)

# Right: 2023 pie chart
ax2 = axes[1]
vals_2023 = df_dockets.loc[df_dockets.year == 2023, cluster_cols].values[0]
labels = [f"{c.title()}\n({v:,})" for c, v in zip(cluster_cols, vals_2023) if v > 0]
sizes = [v for v in vals_2023 if v > 0]
colors_pie = [cluster_colors[c] for c, v in zip(cluster_cols, vals_2023) if v > 0]
ax2.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 9})
ax2.set_title('2023 Docket Breakdown by Cluster', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Year-over-Year Growth: What Changed in 2023?

Let's compute the absolute and percentage change for each cluster between 2022 and 2023 to pinpoint which sectors expanded most.

In [ ]:
row_2022 = df_dockets.loc[df_dockets.year == 2022].iloc[0]
row_2023 = df_dockets.loc[df_dockets.year == 2023].iloc[0]

growth_data = []
for col in cluster_cols:
    v22 = row_2022[col]
    v23 = row_2023[col]
    growth_data.append({
        'Cluster': col.title(),
        '2022 Dockets': int(v22),
        '2023 Dockets': int(v23),
        'Absolute Change': int(v23 - v22),
        '% Change': f"{(v23 - v22) / v22 * 100:.0f}%" if v22 > 0 else 'N/A'
    })

df_growth = pd.DataFrame(growth_data).sort_values('Absolute Change', ascending=False)
print("Year-over-Year Change: 2022 -> 2023")
print("=" * 65)
print(df_growth.to_string(index=False))
print()
print(f"Total change: {int(row_2023['total'] - row_2022['total']):,} additional dockets ({(row_2023['total'] - row_2022['total']) / row_2022['total'] * 100:.0f}% increase)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

df_growth_sorted = df_growth.sort_values('Absolute Change', ascending=True)
colors_bar = [cluster_colors.get(c.lower(), '#999') for c in df_growth_sorted['Cluster']]
ax.barh(df_growth_sorted['Cluster'], df_growth_sorted['Absolute Change'], color=colors_bar)

for i, (_, row) in enumerate(df_growth_sorted.iterrows()):
    ax.text(row['Absolute Change'] + 200, i, f"+{row['Absolute Change']:,} ({row['% Change']})",
            va='center', fontsize=10)

ax.set_title('Absolute Growth in Dockets: 2022 to 2023', fontsize=14, fontweight='bold')
ax.set_xlabel('Additional Dockets in 2023')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

## 4. Deep Dive: The Health Cluster Explosion

The health cluster (FDA, HHS, CMS, CDC, NIH) saw the largest absolute increase. Let's look at health docket trends across all years and compare Biden's health activity to prior administrations.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df_dockets['year'], df_dockets['health'], marker='o', linewidth=2, color='#ef4444', markersize=5)
ax.fill_between(df_dockets['year'], 0, df_dockets['health'], alpha=0.15, color='#ef4444')

# Highlight 2023
health_2023 = df_dockets.loc[df_dockets.year == 2023, 'health'].values[0]
ax.scatter([2023], [health_2023], color='#ef4444', s=150, zorder=5, edgecolors='white', linewidth=2)
ax.annotate(f'{health_2023:,}', xy=(2023, health_2023), xytext=(2020, health_2023 + 1000),
            fontsize=11, fontweight='bold', color='#ef4444',
            arrowprops=dict(arrowstyle='->', color='#ef4444'))

# Admin shading
admin_periods = [
    ('W. Bush', 2001, 2009, '#ef4444', 0.04),
    ('Obama', 2009, 2017, '#3b82f6', 0.04),
    ('Trump', 2017, 2021, '#ef4444', 0.04),
    ('Biden', 2021, 2025, '#3b82f6', 0.06),
]
for name, start, end, color, alpha in admin_periods:
    ax.axvspan(start, end, alpha=alpha, color=color)
    ax.text((start + end) / 2, ax.get_ylim()[1] * 0.95, name, ha='center', fontsize=9, style='italic', alpha=0.6)

ax.set_title('Health Cluster Dockets Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Health Dockets')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

print(f"Health dockets in 2023: {health_2023:,} (peak across entire dataset)")
print(f"Health dockets in 2022: {int(row_2022['health']):,}")
print(f"Growth: {health_2023 - int(row_2022['health']):,} additional health dockets ({(health_2023 - int(row_2022['health'])) / int(row_2022['health']) * 100:.0f}% increase)")

## 5. Top Agencies Driving Biden-Era Docket Volume

Let's identify which specific agencies contributed the most dockets during the Biden administration (2021-2025).

In [ ]:
biden_agencies = df_agency[df_agency.adminId == 'biden'].sort_values('docketCount', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))

colors_agency = [cluster_colors.get(row['cluster'], '#94a3b8') for _, row in biden_agencies.iloc[::-1].iterrows()]
ax.barh(
    [f"{row['agencyCode']} ({row['cluster'].title()})" for _, row in biden_agencies.iloc[::-1].iterrows()],
    biden_agencies.iloc[::-1]['docketCount'],
    color=colors_agency
)

for i, (_, row) in enumerate(biden_agencies.iloc[::-1].iterrows()):
    ax.text(row['docketCount'] + 200, i, f"{row['docketCount']:,}", va='center', fontsize=9)

ax.set_title('Top 15 Agencies by Docket Count (Biden Administration)', fontsize=14, fontweight='bold')
ax.set_xlabel('Docket Count')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

print("\nKey takeaway: FDA alone accounts for 27,172 dockets during Biden's term,")
print("making it by far the largest contributor. FAA (15,911) and EPA (7,809) follow.")

## 6. The Extraordinary 2023 Comment Spike

Beyond docket counts, 2023 also saw a massive spike in public comments: ~386 million, compared to ~12 million in 2022. This suggests one or more highly engaging regulatory proposals drew enormous public participation.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors_comments = ['#ef4444' if y == 2023 else '#6366f1' for y in df_comments['year']]
ax.bar(df_comments['year'], df_comments['total'], color=colors_comments, edgecolor='white', linewidth=0.5)

ax.set_title('Public Comments on Federal Dockets by Year', fontsize=16, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Total Comments')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

comments_2023 = df_comments.loc[df_comments.year == 2023, 'total'].values[0]
ax.annotate(f'2023: {comments_2023/1e6:.0f}M comments', xy=(2023, comments_2023),
            xytext=(2016, comments_2023 * 0.85), fontsize=12, fontweight='bold', color='#ef4444',
            arrowprops=dict(arrowstyle='->', color='#ef4444', lw=1.5))

ax.axvspan(2021, 2025, alpha=0.08, color='blue', label='Biden Admin')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

comments_2022 = df_comments.loc[df_comments.year == 2022, 'total'].values[0]
print(f"2023 comments: {comments_2023:,} ({comments_2023/1e6:.0f}M)")
print(f"2022 comments: {comments_2022:,} ({comments_2022/1e6:.0f}M)")
print(f"Ratio: 2023 had {comments_2023 / comments_2022:.0f}x more comments than 2022")

## 7. Cross-Administration Comparison

How does the Biden administration's regulatory volume compare to prior presidents? Let's compare average annual docket creation rates.

In [ ]:
admin_data = {
    'W. Bush': {'years': range(2001, 2009), 'party': 'R'},
    'Obama': {'years': range(2009, 2017), 'party': 'D'},
    'Trump': {'years': range(2017, 2021), 'party': 'R'},
    'Biden': {'years': range(2021, 2025), 'party': 'D'},
}

comparison = []
for name, info in admin_data.items():
    subset = df_dockets[df_dockets.year.isin(info['years'])]
    comparison.append({
        'Administration': name,
        'Party': info['party'],
        'Years': f"{min(info['years'])}-{max(info['years'])}",
        'Total Dockets': int(subset['total'].sum()),
        'Avg/Year': int(subset['total'].mean()),
        'Peak Year': int(subset.loc[subset.total.idxmax(), 'year']),
        'Peak Value': int(subset['total'].max()),
    })

df_comp = pd.DataFrame(comparison)
print(df_comp.to_string(index=False))
print()

fig, ax = plt.subplots(figsize=(10, 5))
party_colors = {'R': '#ef4444', 'D': '#3b82f6'}
bar_colors = [party_colors[p] for p in df_comp['Party']]
bars = ax.bar(df_comp['Administration'], df_comp['Avg/Year'], color=bar_colors, edgecolor='white')

for bar, val in zip(bars, df_comp['Avg/Year']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'{val:,}/yr', ha='center', fontsize=11, fontweight='bold')

ax.set_title('Average Annual Docket Creation by Administration', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Dockets per Year')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

print(f"\nBiden's average ({df_comp.loc[df_comp.Administration=='Biden', 'Avg/Year'].values[0]:,}/yr) is significantly higher than any prior administration.")
print("This is almost entirely driven by the 2023 anomaly.")

## 8. The "Other" Category: The Largest Absolute Contributor

While health gets attention for its growth rate, the "other" category (agencies not in the 6 named clusters) actually contributed the largest absolute number of dockets in 2023 (19,989). This includes agencies like USCG, FWS, IRS, HUD, MARAD, and many smaller agencies.

In [ ]:
# Compare "other" across years
fig, ax = plt.subplots(figsize=(14, 5))

ax.bar(df_dockets['year'], df_dockets['other'],
       color=['#ef4444' if y == 2023 else '#8b5cf6' for y in df_dockets['year']],
       edgecolor='white', linewidth=0.5)

ax.set_title('"Other" Cluster Dockets by Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Dockets')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

other_2023 = int(df_dockets.loc[df_dockets.year == 2023, 'other'].values[0])
ax.annotate(f'{other_2023:,}', xy=(2023, other_2023), xytext=(2019, other_2023),
            fontsize=11, fontweight='bold', color='#ef4444',
            arrowprops=dict(arrowstyle='->', color='#ef4444'))

plt.tight_layout()
plt.show()

# Top "other" Biden agencies
other_biden = df_agency[(df_agency.adminId == 'biden') & (df_agency.cluster == 'other')].head(10)
print("Top 10 'Other' cluster agencies during Biden administration:")
for _, row in other_biden.iterrows():
    print(f"  {row['agencyCode']:8s} - {row['agencyName']:50s} {row['docketCount']:,}")

## 9. Summary: Why the 2023 Peak?

### Key Findings

1. **Unprecedented docket volume**: 2023 saw 50,279 new dockets -- 2.7x the 2022 level and the highest single year in the 2000-2026 dataset.

2. **Broad-based increase across all clusters**: Every regulatory cluster saw significant growth in 2023, not just one area. This suggests a systemic increase in regulatory activity rather than a single-agency anomaly.

3. **Health cluster led proportionally**: Health dockets tripled from ~4,700 (2022) to ~14,600 (2023), a 213% increase. The FDA alone contributed 27,172 dockets across Biden's full term, far exceeding any other single agency.

4. **"Other" agencies contributed most in absolute terms**: 19,989 dockets came from agencies outside the 6 named clusters, pointing to broad regulatory activity across dozens of smaller agencies.

5. **Comment engagement exploded**: Public comments surged to 386 million in 2023 (vs. 12M in 2022), suggesting one or more highly visible/controversial regulatory proposals drew massive public participation.

6. **Biden administration average was historically high**: At ~28,000 dockets/year, Biden's average was roughly 1.7x the next-highest administration (Obama at ~14,500/year), driven almost entirely by the 2023 spike.

### Likely Contributing Factors

- **Post-COVID regulatory catch-up**: Deferred rulemaking from the pandemic era likely created a backlog that agencies worked through in 2023.
- **Biden executive orders**: The Biden administration issued multiple executive orders directing agencies to pursue new rulemaking across climate, health equity, competition, and technology.
- **FDA activity surge**: Drug approvals, food safety, and tobacco/vaping regulations drove the health cluster spike.
- **Infrastructure Investment and Jobs Act (IIJA) implementation**: Transportation and environment dockets likely increased as agencies implemented the 2021 infrastructure law.
- **Inflation Reduction Act (IRA) implementation**: Energy and environmental regulations tied to IRA provisions began rolling out in 2023.
- **Pre-election regulatory push**: Administrations historically accelerate rulemaking before the final year ("midnight regulations" phenomenon), and 2023 was the year before Biden's final year.

### What This Dashboard Cannot Tell Us

- Which *specific dockets* drove the comment spike (the aggregated data doesn't identify individual dockets)
- Whether the increase reflects genuinely new regulatory activity or data collection improvements in the Mirrulations pipeline
- The impact or significance of individual rules (volume != importance)

To dig deeper, one would need to query the raw `feed_summary.parquet` data for agency-level monthly breakdowns in 2023.

## 10. Next Steps: Querying Raw Data

To go deeper into the 2023 peak, you can query the raw parquet data directly using DuckDB. Here's how to get started:

In [ ]:
# Example: How to query the raw data for monthly 2023 breakdown
# Uncomment to run (requires network access to Cloudflare R2)

# import duckdb
# con = duckdb.connect(":memory:")
# con.execute("INSTALL httpfs; LOAD httpfs; SET s3_region = 'auto';")
#
# R2_URL = "https://pub-5fc11ad134984edf8d9af452dd1849d6.r2.dev/feed_summary.parquet"
#
# # Monthly breakdown for 2023
# result = con.execute(f"""
#     SELECT
#         MONTH(CAST(COALESCE(date_created, modify_date) AS DATE)) AS month,
#         agency_code,
#         COUNT(*) AS docket_count,
#         SUM(comment_count) AS total_comments
#     FROM read_parquet('{R2_URL}')
#     WHERE YEAR(CAST(COALESCE(date_created, modify_date) AS DATE)) = 2023
#     GROUP BY month, agency_code
#     ORDER BY docket_count DESC
# """).fetchdf()
#
# print(result.head(20))
# con.close()

print("Uncomment the code above and run to query the raw Cloudflare R2 parquet data.")
print("This will give you monthly, per-agency breakdowns for 2023.")